# Historical Market Cap Full Downloader

This script downloads full historical market cap data from 1990-01-01 to today for all stocks in the NYSE exchange.
It uses 13-year chunks to maximize efficiency and supports incremental updates.

In [ ]:
import os
import time
import random
import requests
import pandas as pd
from datetime import datetime, timedelta
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv, find_dotenv

# --- CONFIGURATION ---
TARGET_EXCHANGE = "NYSE"
OUTPUT_DIR = "data/historical-market-cap"
GLOBAL_START_DATE = "1990-01-01"
MAX_WORKERS = 10
LIMIT_PER_REQUEST = 5000
CHUNK_SIZE_YEARS = 13
OVERWRITE = False

# --- SETUP ---
load_dotenv(find_dotenv())
api_key = os.getenv("FMP_API_KEY")
base_url = "https://financialmodelingprep.com/stable"
today = datetime.now().strftime('%Y-%m-%d')

def get_date_chunks(start_str, end_str):
    chunks = []
    start_dt = datetime.strptime(start_str, '%Y-%m-%d')
    end_dt = datetime.strptime(end_str, '%Y-%m-%d')
    curr_end = end_dt
    while curr_end >= start_dt:
        curr_start = curr_end - timedelta(days=CHUNK_SIZE_YEARS * 365)
        if curr_start < start_dt:
            curr_start = start_dt
        chunks.append((curr_start.strftime('%Y-%m-%d'), curr_end.strftime('%Y-%m-%d')))
        curr_end = curr_start - timedelta(days=1)
    return chunks

global_chunks = get_date_chunks(GLOBAL_START_DATE, today)

def fetch_mktcap_batch(symbol, start, end, max_retries=5):
    url = f"{base_url}/historical-market-capitalization"
    params = {"symbol": symbol, "from": start, "to": end, "limit": LIMIT_PER_REQUEST, "apikey": api_key}
    for n in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 200:
                data = r.json()
                return pd.DataFrame(data) if data else pd.DataFrame()
            elif r.status_code == 429:
                time.sleep((2 ** (n + 2)) + random.uniform(0, 1))
            else:
                return pd.DataFrame()
        except Exception:
            time.sleep((2 ** (n + 1)) + random.uniform(0, 1))
    return pd.DataFrame()

def process_symbol(symbol):
    safe_symbol = str(symbol).replace(".", "_").replace("/", "_")
    first_char = safe_symbol[0].upper() if safe_symbol[0].isalpha() else "0-9"
    sub_dir = os.path.join(OUTPUT_DIR, first_char)
    os.makedirs(sub_dir, exist_ok=True)
    file_path = os.path.join(sub_dir, f"{safe_symbol}.parquet")
    
    existing_df = pd.DataFrame()
    if os.path.exists(file_path) and not OVERWRITE:
        try:
            existing_df = pd.read_parquet(file_path)
            existing_df['date'] = pd.to_datetime(existing_df['date'])
        except:
            pass

    new_dfs = []
    if existing_df.empty:
        for s, e in global_chunks:
            new_dfs.append(fetch_mktcap_batch(symbol, s, e))
    else:
        min_c = existing_df['date'].min().strftime('%Y-%m-%d')
        max_c = existing_df['date'].max().strftime('%Y-%m-%d')
        if min_c > GLOBAL_START_DATE:
            for s, e in get_date_chunks(GLOBAL_START_DATE, (datetime.strptime(min_c, '%Y-%m-%d') - timedelta(days=1)).strftime('%Y-%m-%d')):
                new_dfs.append(fetch_mktcap_batch(symbol, s, e))
        if max_c < today:
            for s, e in get_date_chunks((datetime.strptime(max_c, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d'), today):
                new_dfs.append(fetch_mktcap_batch(symbol, s, e))
                
    new_dfs = [df for df in new_dfs if not df.empty]
    if new_dfs:
        full_df = pd.concat([existing_df] + new_dfs, ignore_index=True)
        full_df['date'] = pd.to_datetime(full_df['date'])
        full_df = full_df.drop_duplicates(subset=['date']).sort_values('date', ascending=False)
        full_df.to_parquet(file_path, index=False, engine='pyarrow', compression='snappy')
        return f"Updated {symbol}"
    return f"Skip {symbol}"

def fetch_symbols():
    url = f"{base_url}/company-screener"
    params = {"exchange": TARGET_EXCHANGE, "apikey": api_key, "limit": 10000}
    return pd.DataFrame(requests.get(url, params=params).json())['symbol'].unique()

print(f"Fetching symbols...")
symbols = fetch_symbols()
print(f"Starting download for {len(symbols)} tickers...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    list(tqdm(executor.map(process_symbol, symbols), total=len(symbols), desc="Progress"))

In [6]:
df = pd.read_csv("/opt/rws/data/fmp/fmp-company-profiles.csv")

/tmp/ipykernel_822128/2126517208.py:1: DtypeWarning: Columns (0: fullTimeEmployees) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/opt/rws/data/fmp/fmp-company-profiles.csv")


In [9]:
pd.set_option('display.max_columns', None)
df[['symbol', 'exchange']]

,symbol,exchange
0,NVDA,NASDAQ
1,KVYO,NYSE
2,SATS,NASDAQ
3,SHC,NASDAQ
4,STRC,NASDAQ
...,...,...
66681,ALPC,OTC
66682,COPGF,OTC
66683,GCRCF,OTC
66684,CRZNF,OTC
